In [1]:
import json
import numpy as np
import shap
import xgboost as xgb
from sklearn.metrics import accuracy_score

from data_prep import load_titanic

X_train, X_test, y_train, y_test, feature_names = load_titanic()

with open("results.json") as f:
    saved = json.load(f)
best_params = saved["best_params"]

model = xgb.XGBClassifier(
    n_estimators=100, max_depth=best_params["max_depth"], learning_rate=best_params["lr"],
    reg_lambda=best_params["lam"], eval_metric="logloss", random_state=0
)
model.fit(X_train, y_train)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

mean_abs_shap = np.abs(shap_values).mean(axis=0)
ranking = sorted(zip(feature_names, mean_abs_shap), key=lambda t: -t[1])

print("Mean |SHAP value| per feature (impact on model output):")
for name, val in ranking:
    print(f"  {name:<12} {val:.4f}")

# Keep only features with non-negligible SHAP impact, refit, compare.
keep = [name for name, val in ranking if val > 0.02]
keep_idx = [feature_names.index(f) for f in keep]
dropped = [f for f in feature_names if f not in keep]
print(f"\nDropping low-impact features: {dropped}")
print(f"Keeping: {keep}")

model_reduced = xgb.XGBClassifier(
    n_estimators=100, max_depth=best_params["max_depth"], learning_rate=best_params["lr"],
    reg_lambda=best_params["lam"], eval_metric="logloss", random_state=0
)
model_reduced.fit(X_train[:, keep_idx], y_train)
acc_full = accuracy_score(y_test, model.predict(X_test))
acc_reduced = accuracy_score(y_test, model_reduced.predict(X_test[:, keep_idx]))

print(f"\nAccuracy, all {len(feature_names)} features:  {acc_full*100:.1f}%")
print(f"Accuracy, top {len(keep)} SHAP features: {acc_reduced*100:.1f}%")

saved["shap_ranking"] = [[n, float(v)] for n, v in ranking]
saved["results"]["After SHAP feature selection"] = acc_reduced
with open("results.json", "w") as f:
    json.dump(saved, f, indent=2)


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Mean |SHAP value| per feature (impact on model output):
  Sex          1.3258
  Pclass       0.6834
  Fare         0.5030
  Age          0.4192
  Embarked     0.2491
  FamilySize   0.1507
  SibSp        0.0433
  Parch        0.0256
  IsAlone      0.0000

Dropping low-impact features: ['IsAlone']
Keeping: ['Sex', 'Pclass', 'Fare', 'Age', 'Embarked', 'FamilySize', 'SibSp', 'Parch']

Accuracy, all 9 features:  79.9%
Accuracy, top 8 SHAP features: 79.9%
